# Phase 5: Sensitivity Analysis & Business Reporting

This notebook analyzes the results of sensitivity scenarios across cost parameters, service level targets, replenishment lead times, and uncertainty models. It prepares final summary recommendation tables and exports the standardized 15-column schema required for Power BI dashboard reporting.

In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from src.config import Config

## 1. Load Optimization Results & Sensitivity Summary

In [2]:
config = Config('../configs/inventory_config.yaml')
opt_results_path = '../outputs/policy_results/base_case_optimization_results.csv'
sens_results_path = '../outputs/sensitivity_results/sensitivity_summary.csv'

if os.path.exists(opt_results_path):
    opt_df = pd.read_csv(opt_results_path)
    print("--- Base Case Optimization Results ---")
    display(opt_df)
else:
    print("Warning: Base-case results not found.")
    
if os.path.exists(sens_results_path):
    sens_df = pd.read_csv(sens_results_path)
    print("\n--- Sensitivity Summary Shape ---")
    print(sens_df.shape)
else:
    print("Warning: Sensitivity summary not found.")

--- Base Case Optimization Results ---


,family,policy_type,safety_stock,reorder_point,order_quantity,order_up_to_level,fill_rate,cycle_service_level,total_cost,avg_inventory,num_orders,units_short,gamma_benchmark_safety_stock
0,GROCERY I,"(s,Q)",487622.562927,1.750781e+06,314433.578236,NaN,0.991579,0.931267,543439.884926,609857.946773,3.0,36107.045287,486101.043089
1,BEVERAGES,"(s,Q)",588068.725480,1.565681e+06,260348.018483,NaN,0.996574,0.984700,156552.765627,687134.048563,3.0,12874.637348,691113.626070
2,CLEANING,"(R,S)",0.000000,NaN,NaN,798371.817225,1.000000,1.000000,1361.653599,446039.914304,3.0,0.000000,83818.182746



--- Sensitivity Summary Shape ---
(46, 19)


## 2. Analyze Cost Sensitivity

Let's compare the optimal safety stocks and costs across scenarios like different holding rates, order costs, and shortage penalties.

In [3]:
if os.path.exists(sens_results_path):
    # Filter standard cost sensitivity scenarios
    cost_scenarios = ['Base Case', 'Holding Rate 10%', 'Holding Rate 25%', 'Order Cost K=10', 'Order Cost K=50', 'Shortage Cost 50%', 'Shortage Cost 150%']
    cost_df = sens_df[sens_df['scenario'].isin(cost_scenarios)]
    
    for family in config.selected_families:
        fam_df = cost_df[cost_df['family'] == family]
        print(f"\nCost Sensitivity: {family}")
        display(fam_df[['scenario', 'safety_stock', 'reorder_point', 'order_quantity', 'order_up_to_level', 'total_cost', 'fill_rate']])


Cost Sensitivity: GROCERY I


,scenario,safety_stock,reorder_point,order_quantity,order_up_to_level,total_cost,fill_rate
0,Base Case,487622.562927,1.750781e+06,314433.578236,NaN,543439.884926,0.991579
1,Holding Rate 10%,487622.562927,1.750781e+06,317141.927807,NaN,520291.186196,0.991950
2,Holding Rate 25%,487622.562927,1.750781e+06,236395.696446,NaN,679028.765270,0.989361
3,Order Cost K=10,487622.562927,1.750781e+06,193016.277894,NaN,755866.256378,0.988042
4,Order Cost K=50,487622.562927,1.750781e+06,444676.230806,NaN,334013.394360,0.995019
5,Shortage Cost 50%,487622.562927,1.750781e+06,314433.578236,NaN,272637.045271,0.991579
6,Shortage Cost 150%,487622.562927,1.750781e+06,314433.578236,NaN,814242.724581,0.991579



Cost Sensitivity: BEVERAGES


,scenario,safety_stock,reorder_point,order_quantity,order_up_to_level,total_cost,fill_rate
7,Base Case,588068.72548,1.565681e+06,260348.018483,NaN,156552.765627,0.996574
8,Holding Rate 10%,588068.72548,1.565681e+06,259073.669086,NaN,155284.034558,0.996583
9,Holding Rate 25%,588068.72548,1.565681e+06,207966.730084,NaN,177634.224837,0.996098
10,Order Cost K=10,588068.72548,1.565681e+06,169804.124060,NaN,189512.998534,0.995786
11,Order Cost K=50,588068.72548,1.565681e+06,391199.429842,NaN,121149.853153,0.997382
12,Shortage Cost 50%,588068.72548,1.565681e+06,260348.018483,NaN,79304.941537,0.996574
13,Shortage Cost 150%,588068.72548,1.565681e+06,260348.018483,NaN,233800.589716,0.996574



Cost Sensitivity: CLEANING


,scenario,safety_stock,reorder_point,order_quantity,order_up_to_level,total_cost,fill_rate
14,Base Case,0.0,NaN,NaN,798371.817225,1361.653599,1.0
15,Holding Rate 10%,0.0,NaN,NaN,798371.817225,932.769066,1.0
16,Holding Rate 25%,0.0,NaN,NaN,798371.817225,2219.422665,1.0
17,Order Cost K=10,0.0,NaN,NaN,798371.817225,1316.653599,1.0
18,Order Cost K=50,0.0,NaN,NaN,798371.817225,1436.653599,1.0
19,Shortage Cost 50%,0.0,NaN,NaN,798371.817225,1361.653599,1.0
20,Shortage Cost 150%,0.0,NaN,NaN,798371.817225,1361.653599,1.0


## 3. Analyze Service Target Sensitivity

Verify how safety stocks change as the fill rate target $\beta$ varies from $90\%$ to $98\%$.

In [4]:
if os.path.exists(sens_results_path):
    service_df = sens_df[sens_df['scenario'].str.contains('Service Target')]
    for family in config.selected_families:
        fam_df = service_df[service_df['family'] == family]
        print(f"\nService Level Sensitivity: {family}")
        display(fam_df[['target_fill_rate', 'safety_stock', 'total_cost', 'fill_rate']])


Service Level Sensitivity: GROCERY I


,target_fill_rate,safety_stock,total_cost,fill_rate
21,0.90,487622.562927,543439.884926,0.991579
22,0.95,487622.562927,543439.884926,0.991579
23,0.98,487622.562927,543439.884926,0.991579



Service Level Sensitivity: BEVERAGES


,target_fill_rate,safety_stock,total_cost,fill_rate
24,0.90,588068.72548,156552.765627,0.996574
25,0.95,588068.72548,156552.765627,0.996574
26,0.98,588068.72548,156552.765627,0.996574



Service Level Sensitivity: CLEANING


,target_fill_rate,safety_stock,total_cost,fill_rate
27,0.90,0.0,1361.653599,1.0
28,0.95,0.0,1361.653599,1.0
29,0.98,0.0,1361.653599,1.0


## 4. Analyze Lead Time and Review Period Sensitivity

In [5]:
if os.path.exists(sens_results_path):
    lt_df = sens_df[sens_df['scenario'].str.contains('Lead Time')]
    for family in config.selected_families:
        fam_df = lt_df[lt_df['family'] == family]
        print(f"\nLead Time Sensitivity: {family}")
        display(fam_df[['scenario', 'lead_time', 'review_period', 'safety_stock', 'total_cost', 'fill_rate']])


Lead Time Sensitivity: GROCERY I


,scenario,lead_time,review_period,safety_stock,total_cost,fill_rate
30,Lead Time L=1,1,1,487622.562927,543439.884926,0.991579
31,Lead Time L=2,2,1,949468.705873,87153.083474,0.998816



Lead Time Sensitivity: BEVERAGES


,scenario,lead_time,review_period,safety_stock,total_cost,fill_rate
32,Lead Time L=1,1,1,5.880687e+05,156552.765627,0.996574
33,Lead Time L=2,2,1,1.060753e+06,37522.098981,0.999300



Lead Time Sensitivity: CLEANING


,scenario,lead_time,review_period,safety_stock,total_cost,fill_rate
34,"Lead Time L=1, R=1",1,1,0.0,1361.653599,1.000000
35,"Lead Time L=2, R=1",2,1,0.0,1360.334541,0.999999
36,"Lead Time L=2, R=2",2,2,0.0,1360.334541,0.999999


## 5. Verify Standardized Power BI Output Table

Check that the exported Power BI-ready output strictly conforms to the required 15-column schema.

In [6]:
pbi_path = '../outputs/tables/powerbi_inventory_policy_table.csv'
if os.path.exists(pbi_path):
    pbi_df = pd.read_csv(pbi_path)
    print("Power BI Table Columns:")
    print(list(pbi_df.columns))
    print(f"Number of columns: {len(pbi_df.columns)} (Expected: 15)")
    display(pbi_df)
else:
    print("Warning: Power BI standardized table not found.")

Power BI Table Columns:
['family', 'policy_type', 'lead_time', 'review_period', 'risk_period', 'safety_stock', 'reorder_point', 'order_quantity', 'order_up_to_level', 'fill_rate', 'cycle_service_level', 'total_cost', 'avg_inventory', 'num_orders', 'units_short']
Number of columns: 15 (Expected: 15)


,family,policy_type,lead_time,review_period,risk_period,safety_stock,reorder_point,order_quantity,order_up_to_level,fill_rate,cycle_service_level,total_cost,avg_inventory,num_orders,units_short
0,GROCERY I,"(s,Q)",1,NaN,1,487622.5629,1.750781e+06,314433.5782,NaN,0.9916,0.9313,543439.8849,609857.9468,3.0,36107.0453
1,BEVERAGES,"(s,Q)",1,NaN,1,588068.7255,1.565681e+06,260348.0185,NaN,0.9966,0.9847,156552.7656,687134.0486,3.0,12874.6373
2,CLEANING,"(R,S)",1,1.0,2,0.0000,NaN,NaN,798371.8172,1.0000,1.0000,1361.6536,446039.9143,3.0,0.0000
